# PetroOpt AI — Mature Oilfield Production Optimization Prototype

**Purpose:** A beginner-friendly hackathon prototype for monitoring mature wells, forecasting next-day oil production, flagging unusual operating patterns, and comparing illustrative operating scenarios.

**Important:** This notebook generates **synthetic demonstration data** because no organizer dataset was provided here. It is not a validated reservoir simulator, calibrated failure predictor, or operational control system. Do not use its outputs to change real well settings. Replace the synthetic data with approved, documented data before making technical claims.

## 1. Import libraries
Run the cells from top to bottom in Google Colab. No installation should normally be required for these libraries.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
print('PetroOpt AI libraries loaded successfully.')

## 2. Create a demonstration dataset
The data below represents 10 fictional wells over 180 days. The patterns are intentionally simplified for software demonstration, not physical reservoir modelling.

In [ ]:
dates = pd.date_range('2026-01-01', periods=180, freq='D')
records = []

for well_num in range(1, 11):
    well_id = f'W-{well_num:02d}'
    initial_oil = rng.uniform(65, 180)
    initial_pressure = rng.uniform(85, 145)
    initial_water_cut = rng.uniform(25, 65)
    well_factor = rng.uniform(0.85, 1.15)
    for day, date in enumerate(dates):
        pressure = initial_pressure - 0.07 * day + rng.normal(0, 1.3)
        water_cut = np.clip(initial_water_cut + 0.10 * day + rng.normal(0, 1.4), 0, 98)
        # A simple synthetic decline with a mild relationship to pressure and water cut.
        oil_rate = initial_oil * np.exp(-0.0022 * day) * well_factor
        oil_rate *= (max(pressure, 20) / initial_pressure) ** 0.15
        oil_rate *= (1 - 0.002 * max(water_cut - initial_water_cut, 0))
        oil_rate += rng.normal(0, 2.5)
        temperature = 65 + 0.015 * day + rng.normal(0, 1.8)
        vibration = max(0.1, 2.0 + rng.normal(0, 0.22))
        power_kw = max(5, 34 + rng.normal(0, 2.8))
        # Inject a few synthetic abnormal readings for anomaly-detection demonstration.
        if day in (72, 131) and well_num in (3, 7):
            vibration += rng.uniform(1.5, 3.0)
            power_kw += rng.uniform(8, 15)
        records.append({
            'date': date, 'well_id': well_id,
            'oil_rate_bpd': round(max(0, oil_rate), 2),
            'water_cut_pct': round(water_cut, 2),
            'pressure_bar': round(max(1, pressure), 2),
            'temperature_c': round(temperature, 2),
            'pump_vibration': round(vibration, 3),
            'power_kw': round(power_kw, 2)
        })

df = pd.DataFrame(records).sort_values(['well_id', 'date']).reset_index(drop=True)
print(f'Dataset created: {df.shape[0]} rows, {df["well_id"].nunique()} wells')
display(df.head())

## 3. Inspect and understand the data

In [ ]:
print('Shape:', df.shape)
print('\nMissing values per column:')
display(df.isna().sum().to_frame('missing_values'))
print('\nSummary statistics:')
display(df.describe(numeric_only=True).round(2))
print('\nRecords per well:')
display(df.groupby('well_id').size().to_frame('records'))

In [ ]:
daily_field = df.groupby('date', as_index=False).agg(
    total_oil_bpd=('oil_rate_bpd', 'sum'),
    average_water_cut_pct=('water_cut_pct', 'mean'),
    average_pressure_bar=('pressure_bar', 'mean'),
    total_power_kw=('power_kw', 'sum')
)

plt.figure(figsize=(11, 4))
plt.plot(daily_field['date'], daily_field['total_oil_bpd'])
plt.title('Synthetic Field Oil Production Trend')
plt.xlabel('Date'); plt.ylabel('Total oil rate (barrels/day)')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

well_to_plot = 'W-01'
well_df = df[df['well_id'] == well_to_plot]
plt.figure(figsize=(11, 4))
plt.plot(well_df['date'], well_df['oil_rate_bpd'], label='Oil rate')
plt.title(f'Oil Production Trend — {well_to_plot}')
plt.xlabel('Date'); plt.ylabel('Oil rate (barrels/day)')
plt.legend(); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 4. Build a next-day oil-rate forecasting model
We create lag features from each well's own history and split by time (earlier observations for training, later observations for testing). This is more appropriate for time-series forecasting than a random row split. The target is the next day's oil rate.

In [ ]:
model_df = df.copy().sort_values(['well_id', 'date'])
for col in ['oil_rate_bpd', 'water_cut_pct', 'pressure_bar', 'temperature_c', 'pump_vibration', 'power_kw']:
    model_df[f'{col}_lag1'] = model_df.groupby('well_id')[col].shift(1)
    model_df[f'{col}_lag3'] = model_df.groupby('well_id')[col].shift(3)
model_df['target_next_oil_rate_bpd'] = model_df.groupby('well_id')['oil_rate_bpd'].shift(-1)
model_df = model_df.dropna().reset_index(drop=True)

feature_cols = [c for c in model_df.columns if c.endswith('_lag1') or c.endswith('_lag3')]
cutoff_date = model_df['date'].quantile(0.80)
train = model_df[model_df['date'] <= cutoff_date]
test = model_df[model_df['date'] > cutoff_date]
X_train, y_train = train[feature_cols], train['target_next_oil_rate_bpd']
X_test, y_test = test[feature_cols], test['target_next_oil_rate_bpd']

forecast_model = RandomForestRegressor(
    n_estimators=250, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
)
forecast_model.fit(X_train, y_train)
test = test.copy()
test['predicted_next_oil_rate_bpd'] = forecast_model.predict(X_test)

mae = mean_absolute_error(y_test, test['predicted_next_oil_rate_bpd'])
rmse = np.sqrt(mean_squared_error(y_test, test['predicted_next_oil_rate_bpd']))
r2 = r2_score(y_test, test['predicted_next_oil_rate_bpd'])
print(f'Test rows: {len(test)}')
print(f'MAE:  {mae:.2f} barrels/day')
print(f'RMSE: {rmse:.2f} barrels/day')
print(f'R²:   {r2:.3f}')
display(test[['date', 'well_id', 'target_next_oil_rate_bpd', 'predicted_next_oil_rate_bpd']].head(10).round(2))

In [ ]:
sample_well = 'W-01'
sample = test[test['well_id'] == sample_well].sort_values('date')
plt.figure(figsize=(11, 4))
plt.plot(sample['date'], sample['target_next_oil_rate_bpd'], label='Observed next-day rate')
plt.plot(sample['date'], sample['predicted_next_oil_rate_bpd'], label='Model prediction')
plt.title(f'Forecast vs observed — {sample_well} (synthetic test data)')
plt.xlabel('Date'); plt.ylabel('Oil rate (barrels/day)')
plt.legend(); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 5. Flag unusual operating patterns
Isolation Forest assigns an anomaly flag to observations that differ from the broader pattern. This is **not** a calibrated probability of pump failure; flagged points require engineering review.

In [ ]:
anomaly_features = ['oil_rate_bpd', 'water_cut_pct', 'pressure_bar', 'temperature_c', 'pump_vibration', 'power_kw']
anomaly_model = IsolationForest(contamination=0.025, random_state=RANDOM_STATE)
df['anomaly_flag'] = anomaly_model.fit_predict(df[anomaly_features])
df['anomaly_status'] = np.where(df['anomaly_flag'] == -1, 'Review flagged', 'Typical pattern')
anomalies = df[df['anomaly_flag'] == -1].sort_values(['date', 'well_id'])
print(f'Flagged observations: {len(anomalies)} of {len(df)}')
display(anomalies[['date', 'well_id', 'oil_rate_bpd', 'water_cut_pct', 'pressure_bar', 'pump_vibration', 'power_kw', 'anomaly_status']].head(25))

plt.figure(figsize=(8, 4))
plt.scatter(df['pump_vibration'], df['power_kw'], c=df['anomaly_flag'], alpha=0.6)
plt.title('Anomaly screening: vibration vs power (synthetic data)')
plt.xlabel('Pump vibration (synthetic units)'); plt.ylabel('Power (kW)')
plt.tight_layout(); plt.show()

## 6. Compare illustrative operating scenarios
Because this dataset does not contain verified control settings, lift-gas injection, pump curves, nodal-analysis constraints, or a calibrated reservoir model, it cannot calculate safe real-world setpoints. The next cell is only a **toy scenario comparison** to demonstrate how a decision-support table could be presented. The assumed uplift and energy changes are placeholders, not model predictions.

In [ ]:
baseline_oil = float(df.groupby('well_id').tail(1)['oil_rate_bpd'].sum())
baseline_power = float(df.groupby('well_id').tail(1)['power_kw'].sum())

scenario_assumptions = pd.DataFrame([
    {'scenario': 'Baseline', 'assumed_oil_change_pct': 0.0, 'assumed_power_change_pct': 0.0},
    {'scenario': 'Conservative review', 'assumed_oil_change_pct': 1.0, 'assumed_power_change_pct': -2.0},
    {'scenario': 'Balanced review', 'assumed_oil_change_pct': 2.0, 'assumed_power_change_pct': -1.0},
    {'scenario': 'Production-focused review', 'assumed_oil_change_pct': 3.0, 'assumed_power_change_pct': 4.0},
])
scenario_assumptions['illustrative_oil_rate_bpd'] = baseline_oil * (1 + scenario_assumptions['assumed_oil_change_pct'] / 100)
scenario_assumptions['illustrative_power_kw'] = baseline_power * (1 + scenario_assumptions['assumed_power_change_pct'] / 100)
scenario_assumptions['illustrative_power_per_oil_kw_per_bpd'] = scenario_assumptions['illustrative_power_kw'] / scenario_assumptions['illustrative_oil_rate_bpd'].replace(0, np.nan)
display(scenario_assumptions.round(3))
print('These are manually assumed scenarios for UI demonstration, not recommendations or optimized operating setpoints.')

## 7. A simple well-prioritization table
This ranks wells by transparent screening rules (high water cut, low oil rate, and unusual readings). It is a triage aid for the demo, not an automated workover or intervention recommendation.

In [ ]:
latest = df.sort_values('date').groupby('well_id').tail(1).copy()
latest['high_water_cut_points'] = (latest['water_cut_pct'] >= 80).astype(int)
latest['low_oil_rate_points'] = (latest['oil_rate_bpd'] <= latest['oil_rate_bpd'].median()).astype(int)
latest['anomaly_points'] = (latest['anomaly_flag'] == -1).astype(int)
latest['screening_score'] = latest['high_water_cut_points'] + latest['low_oil_rate_points'] + latest['anomaly_points']
priority_table = latest[['well_id', 'oil_rate_bpd', 'water_cut_pct', 'pressure_bar', 'pump_vibration', 'power_kw', 'anomaly_status', 'screening_score']].sort_values('screening_score', ascending=False)
display(priority_table.round(2))
print('Review the underlying measurements and operating context before drawing conclusions.')

## 8. Export demo outputs
These CSV files can be downloaded from the Colab Files panel. They contain synthetic data only.

In [ ]:
df.to_csv('petroopt_synthetic_well_data.csv', index=False)
test[['date', 'well_id', 'target_next_oil_rate_bpd', 'predicted_next_oil_rate_bpd']].to_csv('petroopt_forecast_test_results.csv', index=False)
priority_table.to_csv('petroopt_well_screening_demo.csv', index=False)
scenario_assumptions.to_csv('petroopt_scenario_comparison_demo.csv', index=False)
print('Exported 4 CSV files.')
print('To download in Colab: open the folder icon on the left, locate a CSV, tap its three-dot menu, then Download.')

## 9. How to explain the prototype to judges

- **Problem:** Mature fields can suffer declining production, high water cut, and equipment abnormalities. Manual review can delay investigation.
- **Prototype:** A small Python workflow combines well-level monitoring, next-day oil-rate forecasting, unsupervised anomaly screening, and transparent scenario comparison.
- **Data status:** This version uses synthetic data because verified organizer data was not available when the notebook was created.
- **Validation needed:** Run on approved historical data, use a chronological holdout, compare against a simple baseline, and report errors by well. Confirm anomaly flags with maintenance records.
- **Next development step:** If organizers provide data, map its real column names to the features, check units and missingness, build a leakage-safe time split, and recalibrate the model. Add real lift-gas/VFD optimization only when the necessary engineering constraints and validated models are available.

**Do not claim** that this synthetic demo has improved real production, predicts actual equipment failure, or discovers safe operating setpoints.